# Notebook 09 — Variables categóricas y encoding 🔤

En el NB08 entrenaste tu **primer modelo de Machine Learning** — una regresión lineal para predecir el `fare`. Pero solo usaste **4 features numéricas** (`age`, `pclass`, `sibsp`, `parch`). Te dejaste por fuera información muy valiosa: **¿el pasajero era hombre o mujer? ¿en qué puerto embarcó?**

¿Por qué? Porque scikit-learn **no entiende strings**. Si le pasas `sex="male"`, explota. Hoy aprendes a transformar variables categóricas en algo que el modelo sí entiende: **números**.

## El problema en una imagen

```
    Datos crudos                       Datos para sklearn
    ────────────                       ──────────────────
    sex      → "male"                  sex_male   → 1
    embarked → "S"                     embarked_Q → 0
                                       embarked_S → 1
```

## Objetivos de aprendizaje

1. Entender por qué scikit-learn **no acepta strings** en `X`.
2. Aplicar **one-hot encoding** con `pd.get_dummies()`.
3. Hacer lo mismo con la versión sklearn: `OneHotEncoder`.
4. Distinguir variables **nominales** (sin orden) vs **ordinales** (con orden) — cuándo expandir y cuándo no.
5. Rehacer la regresión del NB08 incluyendo las nuevas features y **comparar el resultado**.

---

## 1. Setup

Cargamos `titanic` y aplicamos la misma limpieza del NB06.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Same cleanup as NB06 / NB08
df = sns.load_dataset("titanic")
df_clean = df.drop(columns=["deck"]).copy()
df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())
df_clean = df_clean.dropna(subset=["embarked"]).reset_index(drop=True)

print(f"df_clean: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
df_clean.head(3)

---

## 2. ¿Por qué scikit-learn no acepta strings?

Internamente, scikit-learn **no compara categorías**, sino que **multiplica números**. Un coeficiente como `β · age` solo tiene sentido si `age` es un número. Con un string, la operación no existe.

La solución es **convertir cada categoría en una columna nueva con 0/1**. Esto se llama **one-hot encoding**.

### Ejemplo conceptual

| Antes (1 columna string) | → | sex_female | sex_male |
|---|---|---|---|
| `"male"`   | → | 0 | 1 |
| `"female"` | → | 1 | 0 |
| `"male"`   | → | 0 | 1 |

Para **N categorías**, se crean **N columnas binarias** (o **N−1** si descartamos una redundante — lo vemos ahora).

### 🔬 Ver el error en vivo

Para que el problema te quede grabado, primero **provoquemos el error** intencionalmente.

In [ ]:
from sklearn.linear_model import LinearRegression

# This will fail because "sex" is a string column
X_broken = df_clean[["age", "sex"]]
y = df_clean["fare"]

try:
    LinearRegression().fit(X_broken, y)
    print("⚠️ El modelo NO debería haberse entrenado con strings...")
except ValueError as e:
    print(f"❌ Error esperado: {type(e).__name__}")
    print(f"   {e}")

---

## 3. Solución 1 — `pd.get_dummies` (la forma más rápida)

`pd.get_dummies` es un atajo de pandas que toma columnas categóricas y las convierte en **columnas binarias** automáticamente.

```python
encoded = pd.get_dummies(
    df_clean,
    columns=["sex", "embarked"],
    drop_first=True,
    dtype=int,
)
```

| Parámetro | Significado |
|---|---|
| `columns=[...]` | Qué columnas codificar (las demás quedan intactas) |
| `drop_first=True` | Descarta la primera categoría de cada columna para evitar redundancia (si `sex_male=0`, ya sabemos que era `female`) |
| `dtype=int` | Las columnas nuevas son enteros 0/1 (por defecto serían `bool`) |

> 💡 **¿Por qué descartar una categoría?** Si tienes `sex_female` y `sex_male`, una columna es **redundante** (siempre suman 1). Esa redundancia confunde a modelos lineales (problema llamado *multicolinealidad*). Para árboles y otros modelos no importa, pero como buena práctica universal usamos `drop_first=True`.

### 🏋️ Ejercicio 1 — Aplicar `get_dummies`

Crea un DataFrame **`df_encoded`** aplicando `pd.get_dummies` a `df_clean`, codificando las columnas `["sex", "embarked"]`, con `drop_first=True` y `dtype=int`.

In [ ]:
# YOUR CODE HERE
df_encoded = None

In [ ]:
# Tests — verify the exercise was completed correctly
assert isinstance(df_encoded, pd.DataFrame), "df_encoded must be a pandas DataFrame"

# Original categorical columns must be gone
assert "sex" not in df_encoded.columns, "Column 'sex' should have been replaced by encoded columns"
assert "embarked" not in df_encoded.columns, "Column 'embarked' should have been replaced by encoded columns"

# New encoded columns must exist (with drop_first=True, 'female' and 'C' are dropped alphabetically)
assert "sex_male" in df_encoded.columns, "Column 'sex_male' is missing"
assert "embarked_Q" in df_encoded.columns, "Column 'embarked_Q' is missing"
assert "embarked_S" in df_encoded.columns, "Column 'embarked_S' is missing"

# Values must be 0/1 integers
for col in ["sex_male", "embarked_Q", "embarked_S"]:
    unique_vals = set(df_encoded[col].unique().tolist())
    assert unique_vals <= {0, 1}, f"Column '{col}' must contain only 0/1, got {unique_vals}"
    assert df_encoded[col].dtype.kind in "iu", f"Column '{col}' must be integer dtype, got {df_encoded[col].dtype}"

# Shape: started with 14 cols (after deck drop), removed 2 (sex, embarked), added 3 → 15
assert df_encoded.shape == (889, 15), f"Expected shape (889, 15), got {df_encoded.shape}"

# Sanity check: row count unchanged
assert len(df_encoded) == len(df_clean), "Row count must not change after encoding"

print("✅ ¡Todos los tests pasaron! Tus variables categóricas ya son numéricas.")
print(f"   df_encoded.shape = {df_encoded.shape}")
encoded_cols = [c for c in df_encoded.columns if c.startswith(("sex_", "embarked_"))]
print(f"   Nuevas columnas:  {encoded_cols}")

---

## 4. Solución 2 — `OneHotEncoder` de scikit-learn

`pd.get_dummies` es ideal para **exploración rápida**. Pero en un flujo de ML serio vas a usar la versión de scikit-learn: **`OneHotEncoder`**. Tiene una ventaja crucial: **encaja en pipelines** y **recuerda las categorías** que vio en `train`, así que cuando llegue el `test` aplica exactamente la misma transformación.

```python
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(drop="first", sparse_output=False, dtype=int)
encoded_array = encoder.fit_transform(df_clean[["sex", "embarked"]])

# .get_feature_names_out() te dice cómo se llaman las columnas resultantes
encoder.get_feature_names_out()
```

| Parámetro | Significado |
|---|---|
| `drop="first"` | Equivalente a `drop_first=True` en `get_dummies` |
| `sparse_output=False` | Devuelve un array denso (más fácil de inspeccionar) |
| `dtype=int` | 0/1 enteros |

> 💡 En el bootcamp aprenderás a meter `OneHotEncoder` dentro de un `Pipeline` para que `train` y `test` se transformen de forma idéntica y automática.

### 🏋️ Ejercicio 2 — `OneHotEncoder` paso a paso

1. Crea **`encoder`** de tipo `OneHotEncoder` con `drop="first"`, `sparse_output=False`, `dtype=int`.
2. Ajústalo y transfórmalo en una sola llamada sobre `df_clean[["sex", "embarked"]]`, guardando el resultado en **`encoded_array`** (un `np.ndarray`).
3. Guarda en **`encoded_names`** los nombres de las columnas resultantes — usa `encoder.get_feature_names_out()`.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# YOUR CODE HERE
encoder = None
encoded_array = None
encoded_names = None

In [ ]:
# Tests — verify the exercise was completed correctly
assert isinstance(encoder, OneHotEncoder), \
    f"encoder must be a OneHotEncoder, got {type(encoder).__name__}"

# Encoder must be fitted
assert hasattr(encoder, "categories_"), "encoder is not fitted — did you call fit_transform?"

# encoded_array must be a 2D numpy array, shape (889, 3): 1 col for sex_male + 2 cols for embarked_Q/S
assert isinstance(encoded_array, np.ndarray), \
    f"encoded_array must be a numpy ndarray, got {type(encoded_array).__name__}"
assert encoded_array.shape == (889, 3), \
    f"encoded_array shape must be (889, 3), got {encoded_array.shape}"

# Values must be only 0/1
unique_vals = set(np.unique(encoded_array).tolist())
assert unique_vals <= {0, 1}, f"encoded_array must contain only 0/1, got {unique_vals}"

# encoded_names should match the columns we'd get from get_dummies
expected_names = {"sex_male", "embarked_Q", "embarked_S"}
assert set(encoded_names) == expected_names, \
    f"encoded_names must be {expected_names}, got {set(encoded_names)}"

print("✅ ¡Todos los tests pasaron! Mismo resultado, otra herramienta.")
print(f"   encoded_array.shape = {encoded_array.shape}")
print(f"   columnas resultantes = {list(encoded_names)}")

---

## 5. Nominal vs ordinal — ¿siempre hay que expandir?

**No.** No todas las variables categóricas son iguales:

| Tipo | ¿Tiene orden natural? | Ejemplo Titanic | ¿Qué hacer? |
|---|---|---|---|
| **Nominal** | No | `sex`, `embarked` (puertos) | **One-hot encoding** |
| **Ordinal** | Sí | `pclass` (1ª > 2ª > 3ª clase) | **Dejarla como está** (ya es numérica con un orden coherente) |

### Caso `pclass`: ya es ordinal

`pclass` toma valores 1, 2, 3. El número **encapsula** el orden (1ª clase > 2ª > 3ª) y la diferencia entre clases es aproximadamente uniforme. **No hace falta one-hot.**

> ⚠️ **Cuidado con el tipo de dato**: si una variable es categórica pero está guardada como número **sin orden real** (por ejemplo, `código_postal = 28013`), tienes que tratarla como nominal aunque sea numérica. Lo importante no es el tipo de dato, sino el **significado**.

---

## 6. Rehacer la regresión del NB08 con las nuevas features

Vamos a comparar:

- **Modelo A (NB08)**: solo 4 features numéricas → `age`, `pclass`, `sibsp`, `parch`.
- **Modelo B (hoy)**: agregamos `sex_male`, `embarked_Q`, `embarked_S` → total 7 features.

Si el encoding aporta información útil, el **R²** debería **subir**.

### 🏋️ Ejercicio 3 — Modelo B con features codificadas

1. Crea **`X_B`** con las columnas `["age", "pclass", "sibsp", "parch", "sex_male", "embarked_Q", "embarked_S"]` de `df_encoded` (en ese orden).
2. Crea **`y_B`** = `df_encoded["fare"]`.
3. Haz un `train_test_split` con `test_size=0.2`, `random_state=42` → `X_train_B`, `X_test_B`, `y_train_B`, `y_test_B`.
4. Entrena un `LinearRegression` → **`model_B`**.
5. Calcula **`r2_B`** = R² entre `y_test_B` y las predicciones del modelo sobre `X_test_B`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# YOUR CODE HERE
X_B = None
y_B = None

X_train_B = None
X_test_B = None
y_train_B = None
y_test_B = None

model_B = None
r2_B = None

In [ ]:
# Tests — verify the exercise was completed correctly
expected_cols = ["age", "pclass", "sibsp", "parch", "sex_male", "embarked_Q", "embarked_S"]
assert list(X_B.columns) == expected_cols, \
    f"X_B columns must be {expected_cols} (in this order), got {list(X_B.columns)}"
assert X_B.shape == (889, 7), f"X_B shape must be (889, 7), got {X_B.shape}"
assert y_B.shape == (889,), f"y_B shape must be (889,), got {y_B.shape}"
assert y_B.name == "fare", f"y_B must come from the 'fare' column, got name '{y_B.name}'"

# Train/test split with random_state=42 must produce specific shapes
assert X_train_B.shape == (711, 7), f"X_train_B shape must be (711, 7), got {X_train_B.shape}"
assert X_test_B.shape == (178, 7), f"X_test_B shape must be (178, 7), got {X_test_B.shape}"
assert y_train_B.shape == (711,), f"y_train_B shape must be (711,), got {y_train_B.shape}"
assert y_test_B.shape == (178,), f"y_test_B shape must be (178,), got {y_test_B.shape}"

# Model must be fitted with 7 coefficients
assert isinstance(model_B, LinearRegression), "model_B must be a LinearRegression instance"
assert hasattr(model_B, "coef_"), "model_B is not fitted — did you call .fit(X_train_B, y_train_B)?"
assert model_B.coef_.shape == (7,), f"Expected 7 coefficients, got {model_B.coef_.shape}"

# r2_B must be a float and should beat the NB08 baseline (~0.30)
assert isinstance(r2_B, float), f"r2_B must be a float, got {type(r2_B).__name__}"
assert r2_B > 0.30, \
    f"r2_B should beat the NB08 baseline (~0.30) thanks to the new features, got {r2_B:.3f}"

print("✅ ¡Todos los tests pasaron! El nuevo modelo está entrenado.")
print(f"   R² (modelo B, 7 features) = {r2_B:.3f}")
print(f"\n   Coeficientes aprendidos:")
for feature, coef in zip(X_B.columns, model_B.coef_):
    print(f"     {feature:15s} {coef:+8.2f}")

---

## 7. Comparación visual de los dos modelos

Para ver la mejora con tus propios ojos, entrenamos también el **modelo A** (idéntico al NB08) y graficamos lado a lado las predicciones de A y B.

In [ ]:
# Train model A (NB08 baseline) for comparison
X_A = df_clean[["age", "pclass", "sibsp", "parch"]]
y_A = df_clean["fare"]
X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(
    X_A, y_A, test_size=0.2, random_state=42
)

model_A = LinearRegression().fit(X_train_A, y_train_A)
predictions_A = model_A.predict(X_test_A)
predictions_B = model_B.predict(X_test_B)

r2_A = r2_score(y_test_A, predictions_A)

print(f"R² modelo A (4 features):              {r2_A:.3f}")
print(f"R² modelo B (7 features con encoding): {r2_B:.3f}")
print(f"Mejora absoluta:                       {(r2_B - r2_A) * 100:+.1f} puntos porcentuales")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, preds, r2, title in [
    (axes[0], predictions_A, r2_A, "Modelo A — 4 features"),
    (axes[1], predictions_B, r2_B, "Modelo B — 7 features (con encoding)"),
]:
    ax.scatter(y_test_A, preds, alpha=0.5)
    lims = [0, max(y_test_A.max(), float(preds.max()))]
    ax.plot(lims, lims, color="red", linestyle="--", label="Predicción perfecta")
    ax.set_xlabel("Fare real")
    ax.set_title(f"{title}\nR² = {r2:.3f}")
    ax.legend()

axes[0].set_ylabel("Fare predicho")
plt.tight_layout()
plt.show()

---

## 8. Resumen — ¿qué aprendiste?

Antes, las variables categóricas eran un **obstáculo** para entrenar un modelo. Ahora son **información útil** que cualquier algoritmo de scikit-learn puede aprovechar.

### Conceptos clave

| Concepto | Idea |
|---|---|
| **One-hot encoding** | Convertir 1 columna categórica en N columnas binarias 0/1 |
| **`pd.get_dummies`** | Atajo de pandas; perfecto para exploración rápida |
| **`OneHotEncoder`** | Versión sklearn; pensada para producción y pipelines |
| **`drop_first=True`** | Descarta una categoría para evitar redundancia |
| **Nominal vs ordinal** | Solo expandir si **no** hay un orden natural en los valores |

### Reglas prácticas

1. **Antes de entrenar** un modelo, revisa `df.dtypes`. Si hay columnas `object` o `category`, hay que codificarlas.
2. **`drop_first=True`** evita columnas redundantes y previene problemas de multicolinealidad en regresión lineal.
3. **`pclass`**, edades, fechas suelen ser **ordinales** — no las expandas alegremente.
4. **`OneHotEncoder` de sklearn** se prefiere en pipelines reales porque "recuerda" las categorías de `train` y aplica la misma transformación al `test`.

### Lo que viene en el NB10

Aprenderás a **inventar features nuevas** que ni siquiera estaban en los datos originales:
- Combinaciones como `family_size = sibsp + parch + 1`.
- Extraer el **título** (Mr, Mrs, Miss, Master) del nombre del pasajero.
- Agrupar la edad en **bins**: niño / joven / adulto / mayor.

Esto se llama **feature engineering** y suele dar más mejora al modelo que cambiar de algoritmo. 💪